In [9]:
# import gurobipy as gp
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objs as go
import plotly.io as pio
import numpy as np
import pandas as pd

Loading Network

In [10]:
def load_edges_from_csv(path_edges: str):
    """
    CSV esperado: node_id1,node_id2,b,f_max
    b = costo base por arco
    f_max = capacidad base
    """
    edges = pd.read_csv(path_edges)
    edges = edges.rename(columns={
        "node_id1": "i",
        "node_id2": "j",
        "b": "c_base",
        "f_max": "U_base"
    })

    edges["e_id"] = np.arange(len(edges))
    return edges

def load_nodes_from_csv(path_nodes: str):
    """
    CSV esperado:
    node_id,d,p_min,p_max,c_var,is_generator,energy_type,instance
    d = demanda base (si no es generador)
    p_max = capacidad gen máxima (si es generador)
    """
    nodes = pd.read_csv(path_nodes)
    nodes = nodes.rename(columns={"node_id":"node"})
    return nodes

Defining the world $W$, the baseline for scenarios

In [11]:
def sample_one_scenario_W(nodes, edges, rng,
                          p_crisis=0.15,
                          # mediocristan
                          dem_sigma=0.10,
                          cost_sigma=0.05,
                          p_outage=0.02,
                          cap_drop=0.7,
                          # extremistan
                          crisis_dem_mult_mu=2.0,
                          crisis_dem_mult_sigma=0.25,

                          crisis_cost_mult=1.25,
                          correlated_outage_nodes=None,
                          correlated_outage_prob=0.8):
    """
    Devuelve un escenario ω completo:
    - D_iω para cada nodo de demanda
    - U_eω para cada arco (capacidad efectiva)
    - c_eω para cada arco (costo operativo)
    """

    # 1) checking if its a crisis scenario
    crisis = rng.random() < p_crisis
        #5% Extremistán
        #95% Mediocristán

    # 2) Demand per node: D_iω
    #Base demand
    D = nodes[["node"]].copy()
    D["D_base"] = nodes["d"].fillna(0.0)

    #Generating demand multiplier (M) based on crisis or not
    if not crisis:
        # Mediocristán: M ~ Normal(1, dem_sigma)
        mult = rng.normal(1.0, dem_sigma, size=len(D))
        mult = np.clip(mult, 0.0, None) # not negative demand
    else:
        # Extremistán: multiplicador grande 
        pareto_alpha = 2.0  # ejemplo
        Y = rng.pareto(a=pareto_alpha, size=len(D))
        mult = 1.0 + Y
    
    # only apply multiplier to demand nodes (D_base > 0)
    D["D_w"] = D["D_base"] * mult
    D.loc[D["D_base"] == 0, "D_w"] = 0.0

    # 3) Cost per arc:
    c = edges[["e_id", "c_base"]].copy()

    if not crisis:
        #mediocristán: small noise around base cost
        # financial uncertainty
        cost_mult = rng.normal(1.0, cost_sigma, size=len(c))
        cost_mult = np.clip(cost_mult, 0.1, None)
    else:
        #extremistán: operative cost increase due to crisis (1.25)
        cost_mult = np.ones(len(c)) * crisis_cost_mult
    c["c_w"] = c["c_base"] * cost_mult

    # 4) Capacity per arc:
    U = edges[["e_id", "U_base", "i", "j"]].copy()

    # mediocristán: each arc has prob p_outage of suffering a capacity reduction (cap_drop)

    outage = rng.random(len(U)) < p_outage
    U_mult = np.where(outage, cap_drop, 1.0)

    #extremistán: if crisis, could happen a "correlated attack" that 
    # completely takes down certain critical arcs (around specific nodes)

    if crisis and correlated_outage_nodes is not None:
        # con prob correlated_outage_prob ocurre "ataque" correlacionado
        if rng.random() < correlated_outage_prob:
            affected = U["i"].isin(correlated_outage_nodes) | U["j"].isin(correlated_outage_nodes)
            # tumba esos arcos completamente
            U_mult = np.where(affected, 0.0, U_mult)

    U["U_w"] = U["U_base"] * U_mult
    
    # 5) Capacidad de generación por nodo generador: G_iω
    gen_nodes = nodes[nodes["is_generator"] == True]
    G = nodes[["node"]].copy()
    G["G_base"] = nodes["p_max"].fillna(0.0)

    if not crisis:
        # Mediocristán: pequeña variación en disponibilidad del generador
        gen_mult = rng.normal(1.0, 0.05, size=len(G))
        gen_mult = np.clip(gen_mult, 0.5, 1.0)  # nunca genera más de su p_max
    else:
        # Extremistán: en crisis generadores pueden caer
        gen_mult = rng.uniform(0.3, 1.0, size=len(G))

    G["G_w"] = G["G_base"] * gen_mult
    G.loc[G["G_base"] == 0, "G_w"] = 0.0  # nodos de demanda no generan

    return {
        "crisis": int(crisis),
        "D": D[["node", "D_w"]],
        "U": U[["e_id", "U_w"]],
        "c": c[["e_id", "c_w"]],
        "G": G[["node", "G_w"]]
    }

In [12]:

def generate_massive_W(nodes, edges, n_samples=1000, seed=123,
                       correlated_outage_nodes=None):
    rng = np.random.default_rng(seed)
    scenarios = []
    for _ in range(n_samples):
        sc = sample_one_scenario_W(nodes, edges, rng,
                                   correlated_outage_nodes=correlated_outage_nodes)
        scenarios.append(sc)
    return scenarios


In [13]:
def scenario_features(sc, nodes, edges):
    """
    Convierte escenario a un vector feature para cluster:
    - demanda total
    - % capacidad perdida (promedio)
    - costo medio
    - indicador crisis
    """
    Dtot = sc["D"]["D_w"].sum()

    U_base = edges.set_index("e_id")["U_base"]
    U_eff  = sc["U"].set_index("e_id")["U_w"]
    cap_loss = 1.0 - (U_eff / U_base.replace(0, np.nan)).fillna(1.0)
    cap_loss_mean = cap_loss.mean()

    c_mean = sc["c"]["c_w"].mean()

    return np.array([Dtot, cap_loss_mean, c_mean, sc["crisis"]], dtype=float)

def reduce_to_K_scenarios(scenarios, nodes, edges, K=20, seed=123):
    """
    Reducción simple:
    - extrae features
    - aplica k-means
    - elige como representante el escenario más cercano al centroide (medoid)
    - prob de cada escenario = proporción de puntos asignados
    """
    from sklearn.cluster import KMeans

    X = np.vstack([scenario_features(sc, nodes, edges) for sc in scenarios])
    km = KMeans(n_clusters=K, random_state=seed, n_init=20)
    labels = km.fit_predict(X)
    centers = km.cluster_centers_

    reps = []
    probs = []
    #encontrar representante de cada cluster
    for k in range(K):
        idx = np.where(labels == k)[0]
        probs.append(len(idx) / len(scenarios))
        # elegir el más cercano al centroide
        dists = np.linalg.norm(X[idx] - centers[k], axis=1)
        rep_idx = idx[np.argmin(dists)]
        reps.append(scenarios[rep_idx])

    return reps, np.array(probs)

In [14]:
# -------------------------
# 5) Exportar W' (20 escenarios) a tablas
# -------------------------

def export_Wprime(reps, probs, out_prefix="Wprime"):
    """
    Crea tres tablas:
    - demanda: (omega, node, D_iw)
    - capacidad: (omega, e_id, U_ew)
    - costos: (omega, e_id, c_ew)
    - prob: (omega, p_omega, crisis)
    """
    rows_D, rows_U, rows_c, rows_p, rows_G = [], [], [], [], []

    for w, (sc, p) in enumerate(zip(reps, probs), start=1):
        rows_p.append({"omega": w, "p_omega": float(p), "crisis": sc["crisis"]})

        for _, r in sc["D"].iterrows():
            rows_D.append({"omega": w, "node": int(r["node"]), "D_iw": float(r["D_w"])})
        for _, r in sc["U"].iterrows():
            rows_U.append({"omega": w, "e_id": int(r["e_id"]), "U_ew": float(r["U_w"])})
        for _, r in sc["c"].iterrows():
            rows_c.append({"omega": w, "e_id": int(r["e_id"]), "c_ew": float(r["c_w"])})
        for _, r in sc["G"].iterrows():
            rows_G.append({"omega": w, "node": int(r["node"]), "G_iw": float(r["G_w"])})

    dfD = pd.DataFrame(rows_D)
    dfU = pd.DataFrame(rows_U)
    dfc = pd.DataFrame(rows_c)
    dfp = pd.DataFrame(rows_p)
    dfG = pd.DataFrame(rows_G)

    dfD.to_csv(f"{out_prefix}_D.csv", index=False)
    dfU.to_csv(f"{out_prefix}_U.csv", index=False)
    dfc.to_csv(f"{out_prefix}_c.csv", index=False)
    dfp.to_csv(f"{out_prefix}_p.csv", index=False)
    dfG.to_csv(f"{out_prefix}_G.csv", index=False)

    return dfD, dfU, dfc, dfp, dfG


In [ ]:
# -------------------------
# Cargar W' para optimización
# -------------------------

# Cargar red y W' para optimización
edges = load_edges_from_csv("edges.csv")
nodes = load_nodes_from_csv("nodes.csv")
instance = 19
nodes = nodes[nodes["instance"] == instance].reset_index(drop=True)
gens = nodes[nodes["is_generator"] == True]["node"].to_list()


# Cargar W' por escenarios
dfD = pd.read_csv("Wprime20_D.csv")
dfU = pd.read_csv("Wprime20_U.csv")
dfc = pd.read_csv("Wprime20_c.csv")
dfp = pd.read_csv("Wprime20_p.csv")
dfG = pd.read_csv("Wprime20_G.csv")

omegas = dfp["omega"].tolist()
p_omega = dict(zip(dfp["omega"], dfp["p_omega"]))

nodes_list = dfD["node"].unique().tolist()
edges_list = dfU["e_id"].unique().tolist()
demand_nodes = dfD[dfD["D_iw"] > 0]["node"].unique().tolist()

# Diccionarios indexados por (omega, id)
D = {(r.omega, r.node): r.D_iw for _, r in dfD.iterrows()}
U_ew = {(r.omega, r.e_id): r.U_ew for _, r in dfU.iterrows()}
c_ew = {(r.omega, r.e_id): r.c_ew for _, r in dfc.iterrows()}
G_iw = {(r.omega, r.node): r.G_iw for _, r in dfG.iterrows()}

# Parámetros de la red base
U_base_dict = dict(zip(edges["e_id"], edges["U_base"]))
delta_U = {e: 0.3 * U_base_dict[e] for e in edges_list}  # +30% al reforzar
H_e = {e: 50.0 for e in edges_list} # costo de hardening
B = 500.0 # presupuesto total
pi = 10000.0 # penalización ENS
alpha = 0.95 # nivel CVaR

# Topología: arcos salientes/entrantes por nodo
out_arcs = {i: edges[edges["i"] == i]["e_id"].tolist() for i in nodes_list}
in_arcs  = {i: edges[edges["j"] == i]["e_id"].tolist() for i in nodes_list}
arc_i = dict(zip(edges["e_id"], edges["i"]))
arc_j = dict(zip(edges["e_id"], edges["j"]))

# Nodos generadores
gen_nodes = nodes[nodes["is_generator"] == True]["node"].tolist()

print("W' cargado:", len(omegas), "escenarios")
print("Nodos:", len(nodes_list), "| Arcos:", len(edges_list))

W' cargado: 20 escenarios
Nodos: 118 | Arcos: 179


Modelo determinístico

In [ ]:
import gurobipy as gp
from gurobipy import GRB

def solve_deterministic(nodes_list, edges_list, demand_nodes, gen_nodes,
                        in_arcs, out_arcs, arc_i, arc_j,
                        U_base_dict, delta_U, H_e, B, pi,
                        D, U_ew, c_ew, G_iw, p_omega, omegas):
    # Calcular promedios ponderados
    D_mean  = {i: sum(p_omega[w] * D[(w, i)] for w in omegas) for i in demand_nodes}
    U_mean  = {e: sum(p_omega[w] * U_ew[(w, e)] for w in omegas) for e in edges_list}
    c_mean  = {e: sum(p_omega[w] * c_ew[(w, e)] for w in omegas) for e in edges_list}
    G_mean  = {i: sum(p_omega[w] * G_iw[(w, i)] for w in omegas) for i in gen_nodes}

    m = gp.Model("Deterministic")
    m.Params.OutputFlag = 0

    h = m.addVars(edges_list, vtype=GRB.BINARY, name="h")
    f = m.addVars(edges_list, lb=0.0, name="f")
    delta_var = m.addVars(demand_nodes, lb=0.0, name="delta")
    g_var = m.addVars(gen_nodes, lb=0.0, name="g")

    m.setObjective(
        gp.quicksum(H_e[e] * h[e] for e in edges_list) +
        gp.quicksum(c_mean[e] * f[e] for e in edges_list) +
        pi * gp.quicksum(delta_var[i] for i in demand_nodes),
        GRB.MINIMIZE
    )

    m.addConstr(gp.quicksum(H_e[e] * h[e] for e in edges_list) <= B, "budget")

    # Balance de flujo
    for i in nodes_list:
        lhs = (gp.quicksum(f[e] for e in in_arcs.get(i, [])) -
               gp.quicksum(f[e] for e in out_arcs.get(i, [])))
        if i in demand_nodes:
            m.addConstr(lhs == D_mean[i] - delta_var[i], f"balance_{i}")
            m.addConstr(delta_var[i] <= D_mean[i], f"shed_ub_{i}")
        elif i in gen_nodes:
            m.addConstr(lhs == -g_var[i], f"balance_{i}")
            m.addConstr(g_var[i] <= G_mean[i], f"gen_ub_{i}")
        else:
            m.addConstr(lhs == 0, f"balance_{i}")

    # Capacidad con hardening
    for e in edges_list:
        cap = U_mean[e] * (U_base_dict[e] + delta_U[e] * h[e])
        m.addConstr(f[e] <= cap, f"cap_{e}")

    m.optimize()

    if m.Status == GRB.OPTIMAL:
        h_sol = {e: round(h[e].X) for e in edges_list}
        obj   = m.ObjVal
        print(f"[Determinístico] Obj = {obj:,.2f} | Líneas reforzadas: {sum(h_sol.values())}")
        return h_sol, obj
    else:
        print("[Determinístico] No se encontró solución óptima")
        return None, None

h_det, obj_det = solve_deterministic(
    nodes_list, edges_list, demand_nodes, gen_nodes,
    in_arcs, out_arcs, arc_i, arc_j,
    U_base_dict, delta_U, H_e, B, pi,
    D, U_ew, c_ew, G_iw, p_omega, omegas
)

Set parameter Username
Set parameter LicenseID to value 2616634


GurobiError: License expired 2026-01-30

Modelo estocástico neutral al riesgo

In [ ]:
def solve_stochastic(nodes_list, edges_list, demand_nodes, gen_nodes,
                     in_arcs, out_arcs, arc_i, arc_j,
                     U_base_dict, delta_U, H_e, B, pi,
                     D, U_ew, c_ew, G_iw, p_omega, omegas):
    m = gp.Model("Stochastic")
    m.Params.OutputFlag = 0

    h = m.addVars(edges_list, vtype=GRB.BINARY, name="h")
    f = m.addVars(edges_list, omegas, lb=0.0, name="f")
    delta_var = m.addVars(demand_nodes, omegas, lb=0.0, name="delta")
    g_var = m.addVars(gen_nodes, omegas, lb=0.0, name="g")

    L = {w: (gp.quicksum(c_ew[(w, e)] * f[e, w] for e in edges_list) +
             pi * gp.quicksum(delta_var[i, w] for i in demand_nodes))
         for w in omegas}

    m.setObjective(
        gp.quicksum(H_e[e] * h[e] for e in edges_list) +
        gp.quicksum(p_omega[w] * L[w] for w in omegas),
        GRB.MINIMIZE
    )

    m.addConstr(gp.quicksum(H_e[e] * h[e] for e in edges_list) <= B, "budget")

    for w in omegas:
        for i in nodes_list:
            lhs = (gp.quicksum(f[e, w] for e in in_arcs.get(i, [])) -
                   gp.quicksum(f[e, w] for e in out_arcs.get(i, [])))
            if i in demand_nodes:
                m.addConstr(lhs == D[(w, i)] - delta_var[i, w], f"bal_{i}_{w}")
                m.addConstr(delta_var[i, w] <= D[(w, i)], f"shed_ub_{i}_{w}")
            elif i in gen_nodes:
                m.addConstr(lhs == -g_var[i, w], f"bal_{i}_{w}")
                m.addConstr(g_var[i, w] <= G_iw[(w, i)], f"gen_ub_{i}_{w}")
            else:
                m.addConstr(lhs == 0, f"bal_{i}_{w}")

        for e in edges_list:
            cap = U_ew[(w, e)] * (U_base_dict[e] + delta_U[e] * h[e])
            m.addConstr(f[e, w] <= cap, f"cap_{e}_{w}")

    m.optimize()

    if m.Status == GRB.OPTIMAL:
        h_sol = {e: round(h[e].X) for e in edges_list}
        obj   = m.ObjVal
        print(f"[Estocástico] Obj = {obj:,.2f} | Líneas reforzadas: {sum(h_sol.values())}")
        return h_sol, obj
    else:
        print("[Estocástico] No se encontró solución óptima")
        return None, None

h_sto, obj_sto = solve_stochastic(
    nodes_list, edges_list, demand_nodes, gen_nodes,
    in_arcs, out_arcs, arc_i, arc_j,
    U_base_dict, delta_U, H_e, B, pi,
    D, U_ew, c_ew, G_iw, p_omega, omegas
)

Modelo con aversión al riesgo

In [ ]:
def solve_cvar(nodes_list, edges_list, demand_nodes, gen_nodes,
               in_arcs, out_arcs, arc_i, arc_j,
               U_base_dict, delta_U, H_e, B, pi,
               D, U_ew, c_ew, G_iw, p_omega, omegas,
               alpha=0.95, lam=0.5):
    m = gp.Model("CVaR")
    m.Params.OutputFlag = 0

    h = m.addVars(edges_list, vtype=GRB.BINARY, name="h")
    f = m.addVars(edges_list, omegas, lb=0.0, name="f")
    delta_var = m.addVars(demand_nodes, omegas, lb=0.0, name="delta")
    g_var = m.addVars(gen_nodes, omegas, lb=0.0, name="g")
    eta = m.addVar(lb=-GRB.INFINITY, name="eta")
    s   = m.addVars(omegas, lb=0.0, name="s")

    L = {w: (gp.quicksum(c_ew[(w, e)] * f[e, w] for e in edges_list) +
             pi * gp.quicksum(delta_var[i, w] for i in demand_nodes))
         for w in omegas}

    cvar_expr = eta + (1.0 / (1.0 - alpha)) * gp.quicksum(p_omega[w] * s[w] for w in omegas)
    exp_cost  = gp.quicksum(p_omega[w] * L[w] for w in omegas)
    invest    = gp.quicksum(H_e[e] * h[e] for e in edges_list)

    m.setObjective(
        invest + (1 - lam) * exp_cost + lam * cvar_expr,
        GRB.MINIMIZE
    )

    m.addConstr(gp.quicksum(H_e[e] * h[e] for e in edges_list) <= B, "budget")

    for w in omegas:
        m.addConstr(s[w] >= L[w] - eta, f"cvar_s_{w}")

        for i in nodes_list:
            lhs = (gp.quicksum(f[e, w] for e in in_arcs.get(i, [])) -
                   gp.quicksum(f[e, w] for e in out_arcs.get(i, [])))
            if i in demand_nodes:
                m.addConstr(lhs == D[(w, i)] - delta_var[i, w], f"bal_{i}_{w}")
                m.addConstr(delta_var[i, w] <= D[(w, i)], f"shed_ub_{i}_{w}")
            elif i in gen_nodes:
                m.addConstr(lhs == -g_var[i, w], f"bal_{i}_{w}")
                m.addConstr(g_var[i, w] <= G_iw[(w, i)], f"gen_ub_{i}_{w}")
            else:
                m.addConstr(lhs == 0, f"bal_{i}_{w}")

        for e in edges_list:
            cap = U_ew[(w, e)] * (U_base_dict[e] + delta_U[e] * h[e])
            m.addConstr(f[e, w] <= cap, f"cap_{e}_{w}")

    m.optimize()

    if m.Status == GRB.OPTIMAL:
        h_sol = {e: round(h[e].X) for e in edges_list}
        obj   = m.ObjVal
        print(f"[CVaR alpha={alpha}] Obj = {obj:,.2f} | Líneas reforzadas: {sum(h_sol.values())}")
        return h_sol, obj
    else:
        print("[CVaR] No se encontró solución óptima")
        return None, None

h_cvar, obj_cvar = solve_cvar(
    nodes_list, edges_list, demand_nodes, gen_nodes,
    in_arcs, out_arcs, arc_i, arc_j,
    U_base_dict, delta_U, H_e, B, pi,
    D, U_ew, c_ew, G_iw, p_omega, omegas,
    alpha=alpha, lam=0.5
)

Backtesting

In [ ]:
def backtest(h_sol, nodes_list, edges_list, demand_nodes, gen_nodes,
             in_arcs, out_arcs, U_base_dict, delta_U,
             pi, nodes, edges, n_test=10, seed=99):
    rng = np.random.default_rng(seed)
    costos = []

    for _ in range(n_test):
        sc = sample_one_scenario_W(nodes, edges, rng,
                                   correlated_outage_nodes=gens)

        U_sc = dict(zip(sc["U"]["e_id"], sc["U"]["U_w"]))
        c_sc = dict(zip(sc["c"]["e_id"], sc["c"]["c_w"]))
        D_sc = dict(zip(sc["D"]["node"], sc["D"]["D_w"]))
        G_sc = dict(zip(sc["G"]["node"], sc["G"]["G_w"]))

        m = gp.Model("backtest_op")
        m.Params.OutputFlag = 0

        f  = m.addVars(edges_list, lb=0.0, name="f")
        dv = m.addVars(demand_nodes, lb=0.0, name="delta")
        gv = m.addVars(gen_nodes, lb=0.0, name="g")

        m.setObjective(
            gp.quicksum(c_sc[e] * f[e] for e in edges_list) +
            pi * gp.quicksum(dv[i] for i in demand_nodes),
            GRB.MINIMIZE
        )

        for i in nodes_list:
            lhs = (gp.quicksum(f[e] for e in in_arcs.get(i, [])) -
                   gp.quicksum(f[e] for e in out_arcs.get(i, [])))
            if i in demand_nodes:
                d_i = D_sc.get(i, 0.0)
                m.addConstr(lhs == d_i - dv[i], f"bal_{i}")
                m.addConstr(dv[i] <= d_i, f"shed_ub_{i}")
            elif i in gen_nodes:
                m.addConstr(lhs == -gv[i], f"bal_{i}")
                m.addConstr(gv[i] <= G_sc.get(i, 0.0), f"gen_ub_{i}")
            else:
                m.addConstr(lhs == 0, f"bal_{i}")

        for e in edges_list:
            cap = U_sc[e] * (U_base_dict[e] + delta_U[e] * h_sol.get(e, 0))
            m.addConstr(f[e] <= cap, f"cap_{e}")

        m.optimize()

        if m.Status == GRB.OPTIMAL:
            invest_cost = sum(H_e[e] * h_sol.get(e, 0) for e in edges_list)
            costos.append(invest_cost + m.ObjVal)
        else:
            costos.append(np.nan)

    return np.array(costos)

print("Backtesting escenarios de prueba...")
costos_det  = backtest(h_det,  nodes_list, edges_list, demand_nodes, gen_nodes, in_arcs, out_arcs, U_base_dict, delta_U, pi, nodes, edges, seed=99)
costos_sto  = backtest(h_sto,  nodes_list, edges_list, demand_nodes, gen_nodes, in_arcs, out_arcs, U_base_dict, delta_U, pi, nodes, edges, seed=100)
costos_cvar = backtest(h_cvar, nodes_list, edges_list, demand_nodes, gen_nodes, in_arcs, out_arcs, U_base_dict, delta_U, pi, nodes, edges, seed=101)
print("Backtesting completado.")

Metricas y comparación

In [ ]:
def metricas(costos, nombre, alpha=0.95):
    costos = costos[~np.isnan(costos)]
    mean   = np.mean(costos)
    p90    = np.percentile(costos, 90)
    p95    = np.percentile(costos, 95)
    p99    = np.percentile(costos, 99)
    # CVaR empírico
    var    = np.percentile(costos, alpha * 100)
    cvar   = costos[costos >= var].mean()
    return {"Modelo": nombre, "E[Costo]": mean,
            "P90": p90, "P95": p95, "P99": p99, "CVaR_95": cvar}

rows = [
    metricas(costos_det,  "Determinístico"),
    metricas(costos_sto,  "Estocástico"),
    metricas(costos_cvar, "CVaR"),
]
df_metricas = pd.DataFrame(rows).set_index("Modelo")
print(df_metricas.to_string())

Gráficos

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Boxplot de costos
axes[0].boxplot(
    [costos_det[~np.isnan(costos_det)],
     costos_sto[~np.isnan(costos_sto)],
     costos_cvar[~np.isnan(costos_cvar)]],
    labels=["Determinístico", "Estocástico", "CVaR"],
    patch_artist=True,
    boxprops=dict(facecolor="#AED6F1"),
    medianprops=dict(color="navy", linewidth=2)
)
axes[0].set_title("Distribución de costos totales (backtesting)")
axes[0].set_ylabel("Costo total")
axes[0].grid(axis="y", alpha=0.4)

# Histograma superpuesto
for costos, label, color in [
    (costos_det,  "Determinístico", "steelblue"),
    (costos_sto,  "Estocástico",    "darkorange"),
    (costos_cvar, "CVaR",           "seagreen"),
]:
    c = costos[~np.isnan(costos)]
    axes[1].hist(c, bins=40, alpha=0.5, label=label, color=color, density=True)

axes[1].set_title("Histograma de costos (backtesting)")
axes[1].set_xlabel("Costo total")
axes[1].set_ylabel("Densidad")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig("comparacion_modelos.png", dpi=150)
plt.show()

# Tabla de métricas visualizada
fig2, ax2 = plt.subplots(figsize=(9, 2))
ax2.axis("off")
tabla = df_metricas.reset_index()
t = ax2.table(
    cellText=tabla.values.tolist(),
    colLabels=tabla.columns.tolist(),
    cellLoc="center", loc="center"
)
t.auto_set_font_size(False)
t.set_fontsize(10)
t.scale(1.2, 1.6)
plt.title("Métricas comparativas de los tres modelos", pad=12)
plt.tight_layout()
plt.savefig("tabla_metricas.png", dpi=150)
plt.show()

In [12]:
# -------------------------
# 6) Ejemplo de uso
# -------------------------

edges = load_edges_from_csv("edges.csv")
nodes = load_nodes_from_csv("nodes.csv")
edges
# 1) W masivo (mundo verdadero)
escenarios_W = generate_massive_W(nodes, edges, n_samples=1000, seed=1,
                                  correlated_outage_nodes=[9,4])  # ejemplo: "zona crítica"

# 2) reducir a 20 (W' = hatW)
reps, probs = reduce_to_K_scenarios(escenarios_W, nodes, edges, K=20, seed=1)

# 3) exportar tablas para optimización
dfD, dfU, dfc, dfp = export_Wprime(reps, probs, out_prefix="Wprime20")

In [15]:
print(dfD)

        omega  node        D_iw
0           1     0   29.653743
1           1     1   11.610565
2           1     2   23.974939
3           1     3   17.121625
4           1     4    0.000000
...       ...   ...         ...
226555     20   113    4.638790
226556     20   114   15.118954
226557     20   115  111.245778
226558     20   116    9.813908
226559     20   117   17.348178

[226560 rows x 3 columns]
